# Practice Dijkstra's with the Movies dataset

Before tackling real logistics data, we'll master Dijkstra's on the familiar 
Movies dataset.

**Why this approach?**
- You already know the data (actors, movies, relationships)
- Easier to verify results ("Does this path make sense?")
- Same algorithm, simpler mental model

**What you'll learn:**
- How Dijkstra's finds shortest weighted paths
- The difference between hop count and weighted cost
- How to invert weights for "strongest path" problems
- Stream, mutate, and write execution modes

By the end, you'll be ready to find optimal shipping routes in the logistics network.

## Imports

In [ ]:
import os
from datetime import timedelta
from IPython.display import display
from dotenv import load_dotenv
from graphdatascience.session import GdsSessions, AuraAPICredentials, DbmsConnectionInfo, SessionMemory

## Setup: Connect to AGA

Let's set up our connection to Aura Graph Analytics.

In [ ]:
# Load environment variables
load_dotenv()

# Get Aura API credentials
client_id = os.getenv('AURA_CLIENT_ID')
client_secret = os.getenv('AURA_CLIENT_SECRET')
project_id = os.getenv('AURA_PROJECT_ID')  # set in .env only if your Aura account has multiple projects

# Get AuraDB connection info
uri = os.getenv('AURA_URI')
username = os.getenv('AURA_USERNAME')
password = os.getenv('AURA_PASSWORD')

In [ ]:
# Create sessions manager
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(client_id, client_secret, project_id=project_id)
)

In [ ]:
# Create a GDS Session
gds = sessions.get_or_create(
    session_name="movies-dijkstras",
    memory=SessionMemory.m_2GB,
    db_connection=DbmsConnectionInfo(
        uri=uri,
        username=username,
        password=password
    ),
    ttl=timedelta(minutes=30)
)
print(f"Client ID starts with: {client_id[:12]}...")

# Verify connection
gds.verify_connectivity()
print(f"Connected to GDS Session: movies-dijkstras")

In [ ]:
from workshop_helpers import configure, visualize_query, visualize_projection

configure(gds, uri, username, password, os.getenv("AURA_DATABASE") or "neo4j")
print("Helper functions loaded: visualize_query(), visualize_projection()")

## Practice: Project the Movie Graph

Let's find the shortest path between actors through their collaborations. First, project the graph.

This creates an undirected graph where actors are connected if they've appeared in the same movie. The `collaborations` property counts how many movies they've shared.

In [ ]:
# Project actor collaborations using remote projection
G_collab, result = gds.graph.project(
    "actor-collaborations",
    """
    CALL {
      MATCH (a1)-[r:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor)
      WHERE a1 <> a2
      RETURN a1 AS source, 
            a2 AS target, 
            type(r) AS relType,
            a1{} AS sourceNodeProperties,
            a2{} AS targetNodeProperties,
            count(r) AS collaborations
    }
    RETURN gds.graph.project.remote(source, target, {
      sourceNodeLabels: labels(source),
      targetNodeLabels: labels(target),
      sourceNodeProperties: sourceNodeProperties,
      targetNodeProperties: targetNodeProperties,
      relationshipType: 'COLLABORATED',
      relationshipProperties: {collaborations: collaborations}
    })
    """
)

print(f"Projected graph: {G_collab.name()}")
print(f"  Nodes: {G_collab.node_count():,}")
print(f"  Relationships: {G_collab.relationship_count():,}")

In [ ]:
VG = visualize_projection(G_collab)
VG.render()

## Find the Shortest Path

Now, let's find the shortest path between two actors.

Without a weight property, this finds the path with fewest hops—the classic "degrees of separation."

In [ ]:
# Find source and target node IDs by label and property
source_id = gds.find_node_id(["Actor"], {"name": "Shah Rukh Khan"})
target_id = gds.find_node_id(["Actor"], {"name": "Lee Jung-jae"})

print(f"Source: Shah Rukh Khan (ID: {source_id})")
print(f"Target: Lee Jung-jae (ID: {target_id})")

In [ ]:
# Run Dijkstra's algorithm
result = gds.shortestPath.dijkstra.stream(
    G_collab,
    sourceNode=source_id,
    targetNode=target_id
)

# Get path node IDs from the first row
path_node_ids = result['nodeIds'].iloc[0]

# Resolve to actor names
gds.run_cypher("""
    UNWIND $nodeIds AS nodeId
    RETURN gds.util.asNode(nodeId).name AS name
""", params={"nodeIds": list(path_node_ids)})

## Single Target vs Multiple Targets

Dijkstra's can find paths to **multiple targets** in a single call. This is more efficient than running separate queries for each target.

In [ ]:
# Find source and target node IDs
source_id = gds.find_node_id(["Actor"], {"name": "Shah Rukh Khan"})
target_ids = [
    gds.find_node_id(["Actor"], {"name": "Lee Jung-jae"}),
    gds.find_node_id(["Actor"], {"name": "Joe Pantoliano"}),
    gds.find_node_id(["Actor"], {"name": "Keanu Reeves"})
]

print(f"Source: Shah Rukh Khan")
print(f"Targets: Lee Jung-jae, Joe Pantoliano, Keanu Reeves")

In [ ]:
# Run Dijkstra's with multiple targets
multi_result = gds.shortestPath.dijkstra.stream(
    G_collab,
    sourceNode=source_id,
    targetNodes=target_ids
)

# Display results for each target
for idx, row in multi_result.iterrows():
    path_names = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(row['nodeIds'])})
    
    print(f"\nTarget: {path_names['name'].iloc[-1]}")
    print(f"  Hops: {int(row['totalCost'])}")
    print(f"  Path: {' -> '.join(path_names['name'].tolist())}")

## Dijkstra's Finds the Minimum Cost

We projected collaboration counts to the graph too. Let's add those as a weight and see what happens.

The algorithm is now biased to choose the shortest path between the characters who have collaborated the **least**.

This happens because Dijkstra's always interprets relationship weights as costs.

In [ ]:
# Run Dijkstra's with collaboration weights
weighted_result = gds.shortestPath.dijkstra.stream(
    G_collab,
    sourceNode=source_id,
    targetNodes=target_ids,
    relationshipWeightProperty='collaborations'
)

# Display results
for idx, row in weighted_result.iterrows():
    path_names = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(row['nodeIds'])})
    
    print(f"\nTarget: {path_names['name'].iloc[-1]}")
    print(f"  Total Cost: {int(row['totalCost'])}")
    print(f"  Path: {' -> '.join(path_names['name'].tolist())}")

### Visualize: Collaboration Paths (No Weights)

Let's visualize the paths between actors as a graph.

In [ ]:
# Helper function to visualize Dijkstra paths
def visualize_dijkstra_paths(result, title="Dijkstra's Paths"):
    """Visualize paths from Dijkstra's result using visualize_query."""
    # Collect all actor names from all paths
    all_node_ids = set()
    for idx, row in result.iterrows():
        all_node_ids.update(row['nodeIds'])
    
    # Get actor names for the query
    names_df = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(all_node_ids)})
    names = names_df['name'].tolist()
    
    # Build Cypher query to match paths between these actors
    query = f"""
        MATCH path = (a1:Actor)-[r1:ACTED_IN]->(m:Movie)<-[r2:ACTED_IN]-(a2:Actor)
        WHERE a1.name IN {names} AND a2.name IN {names} AND a1 <> a2
        RETURN path
    """
    
    print(title)
    return visualize_query(query)

In [ ]:
# Visualize unweighted paths
unweighted_result = gds.shortestPath.dijkstra.stream(
    G_collab,
    sourceNode=source_id,
    targetNodes=target_ids
)

VG = visualize_dijkstra_paths(unweighted_result, "Dijkstra's Paths (No Weights - Fewest Hops)")
VG.render()

In [ ]:
# Visualize weighted paths
VG = visualize_dijkstra_paths(weighted_result, "Dijkstra's Paths (Weighted - Least Collaborations)")
VG.render()

In [ ]:
# Project with inverted weights using remote projection
G_weighted, result = gds.graph.project(
    "actor-weighted",
    """
    CALL {
        MATCH (a1:Actor)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor)
        WHERE a1 <> a2
        WITH a1, a2, count(m) AS collaborations
        RETURN 
          a1 AS source, 
          a2 AS target,
          'COLLABORATED' AS relType,
          a1{} AS sourceNodeProperties,
          a2{} AS targetNodeProperties,
          1.0 / collaborations AS invertedCollab
    }
    RETURN gds.graph.project.remote(source, target, {
      sourceNodeLabels: labels(source),
      targetNodeLabels: labels(target),
      sourceNodeProperties: sourceNodeProperties,
      targetNodeProperties: targetNodeProperties,
      relationshipType: relType,
      relationshipProperties: {invertedCollab: invertedCollab}
    })
    """
)

print(f"Projected graph: {G_weighted.name()}")
print(f"  Nodes: {G_weighted.node_count():,}")
print(f"  Relationships: {G_weighted.relationship_count():,}")

In [ ]:
VG = visualize_projection(G_weighted)
VG.render()

## Find Strongest Collaboration Paths

Now, let's run the same query with our inverted weights and see what happens.

In [ ]:
# Find strongest collaboration paths
inverted_result = gds.shortestPath.dijkstra.stream(
    G_weighted,
    sourceNode=source_id,
    targetNodes=target_ids,
    relationshipWeightProperty='invertedCollab'
)

# Display results
for idx, row in inverted_result.iterrows():
    path_names = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(row['nodeIds'])})
    
    print(f"\nTarget: {path_names['name'].iloc[-1]}")
    print(f"  Path: {' -> '.join(path_names['name'].tolist())}")

### Visualize: Strongest Collaboration Paths

The previous query returned yet another set of shortest paths. Along the path, any two connections are the **strongest** collaborators on the shortest path.

In [ ]:
# Visualize strongest collaboration paths
VG = visualize_dijkstra_paths(inverted_result, "Dijkstra's Paths (Inverted Weights - Strongest Collaborations)")
VG.render()

## Write Mode

Write mode persists the path as a new relationship in Neo4j.

In [ ]:
# Get Keanu Reeves node ID
keanu_id = gds.find_node_id(["Actor"], {"name": "Keanu Reeves"})

# Write the shortest path as a relationship
write_result = gds.shortestPath.dijkstra.write(
    G_collab,
    sourceNode=source_id,
    targetNode=keanu_id,
    writeRelationshipType='SHORTEST_PATH',
    writeNodeIds=True,
    writeCosts=True
)

print(f"Relationships written: {write_result['relationshipsWritten']}")

## Write Mode Configuration

Additional parameters for write mode:

| Parameter | Type | Description |
|-----------|------|-------------|
| writeRelationshipType | String | Type for the new relationship (required) |
| writeNodeIds | Boolean | Store intermediate node IDs on relationship |
| writeCosts | Boolean | Store accumulated costs on relationship |

The written relationship connects source directly to target, with path details stored as properties.

In [ ]:
# See the relationship we just created
shortest_path_rel = gds.run_cypher("""
    MATCH (a:Actor {name: 'Shah Rukh Khan'})-[r:SHORTEST_PATH]->(a2:Actor {name: 'Keanu Reeves'})
    RETURN a.name AS actor1, r.totalCost AS totalCost, r.nodeIds AS nodeIds, r.costs AS costs, a2.name AS actor2
""")

display(shortest_path_rel)

In [ ]:
# Visualize the written SHORTEST_PATH relationship
VG = visualize_query("""
    MATCH path = (a:Actor {name: 'Shah Rukh Khan'})-[r:SHORTEST_PATH]->(a2:Actor {name: 'Keanu Reeves'})
    RETURN path
""")
VG.render()

## Mutate Mode

Mutate mode adds the path to the **in-memory graph** without writing to Neo4j.

Useful when chaining multiple algorithms together.

Note: The relationship produced is always **directed**, even if the input graph is undirected. This is because a path has a natural direction from source to target.

In [ ]:
# Get Tom Hanks node ID
tom_id = gds.run_cypher("""
    MATCH (a:Actor {name: 'Tom Hanks'})
    RETURN id(a) AS id
""")['id'].iloc[0]

# Mutate - add path to in-memory graph only
mutate_result = gds.shortestPath.dijkstra.mutate(
    G_collab,
    sourceNode=tom_id,
    targetNode=keanu_id,
    mutateRelationshipType='SHORTEST_PATH'
)

print(f"Relationships written to projection: {mutate_result['relationshipsWritten']}")

## Performance Considerations

Dijkstra's is efficient:

* **Time complexity:** O((V + E) log V)
* **Space complexity:** O(V) for the priority queue
* **Single-threaded:** Changing concurrency has no effect

The projection step is typically the bottleneck, not the algorithm itself.

## When Dijkstra's Won't Help

Dijkstra's is **not** the right choice when:

* You need paths with **negative weights** → Use Bellman-Ford
* You need **all** shortest paths, not just one → Use All Pairs Shortest Path
* You need **k shortest paths** → Use Yen's algorithm
* Weights are all equal → Native Cypher SHORTEST is simpler

## Relationship Type Filtering

You can filter which relationship types to traverse by specifying them in the algorithm call.

In [ ]:
# Example: Filter by relationship types
# (In our actor-collaborations graph, we only have one type, but here's the syntax)

filtered_result = gds.shortestPath.dijkstra.stream(
    G_collab,
    sourceNode=source_id,
    targetNode=keanu_id,
    relationshipTypes=['*']  # Use specific types like ['COLLABORATED'] if available
)

print(f"Path found with {filtered_result['totalCost'].iloc[0]} hops")

In [ ]:
# Filter by relationship types
filtered_result = gds.shortestPath.dijkstra.stream(
    G_collab,
    sourceNode=source_id,
    targetNode=keanu_id,
    relationshipTypes=['COLLABORATED']  # Only traverse these types
)

print(f"Path found with {int(filtered_result['totalCost'].iloc[0])} hops")

In [ ]:
# Filter by node labels
filtered_result = gds.shortestPath.dijkstra.stream(
    G_collab,
    sourceNode=source_id,
    targetNode=keanu_id,
    nodeLabels=['Actor']  # Only traverse nodes with these labels
)

print(f"Path found with {int(filtered_result['totalCost'].iloc[0])} hops")

## Dijkstra's vs Other Pathfinding

| Algorithm | Finds | Use When |
|-----------|-------|----------|
| **Dijkstra's** | Single shortest path | You need one optimal route |
| **Yen's** | K shortest paths | You need alternatives |
| **A*** | Single shortest path | You have a heuristic (e.g., geographic distance) |
| **Bellman-Ford** | Single shortest path | You have negative weights |

## Common Use Cases

Dijkstra's is widely used for:

* **Logistics** — Finding fastest shipping routes
* **Navigation** — GPS and mapping applications
* **Network routing** — Internet packet delivery
* **Game AI** — Character pathfinding
* **Social networks** — Degrees of separation (like our actor example!)

Any network where you need the optimal weighted path is a candidate.

## Transfer: From Movies to Logistics

You've now learned Dijkstra's on the Movie graph:

| Movies | Logistics |
|--------|----------|
| Actors | Locations (airports, warehouses) |
| Collaboration strength | Transit time |
| Degrees of separation | Optimal route |

In the next lesson, you'll apply this to the logistics network—finding optimal shipping routes and comparing them to historical operations.

## Clean Up

Drop the practice projections.

In [ ]:
gds.run_cypher("MATCH ()-[r:SHORTEST_PATH]->() DELETE r")

In [ ]:
# Drop all projections
for graph_name in gds.graph.list()["graphName"].tolist():
    gds.graph.drop(graph_name)
    print(f"Dropped: {graph_name}")

In [ ]:
# Delete the session
gds.delete()
print("Session deleted - billing stopped")

## Summary

You've mastered Dijkstra's on the Movies dataset:

| What You Learned | How It Applies to Logistics |
|------------------|----------------------------|
| Finding paths between actors | Finding routes between airports |
| Collaboration count as weight | Transit time as weight |
| Inverting weights for "strongest" | (Not needed - we want minimum time) |
| Multiple targets in one call | Finding routes to multiple destinations |
| Write mode for persistence | Saving recommended routes |

**Key takeaway:** Dijkstra's finds the ONE best path. It minimizes total cost 
(sum of weights along the path).

**Next notebook:** Apply this to the Cargo 2000 logistics network. We'll find 
optimal shipping routes and see how they compare to what we've historically done.